In [20]:
import json
import os
from tqdm import tqdm

In [21]:
def load_json(file_path):
    with open(file_path, 'r', encoding='utf-8') as file:
        return json.load(file)

def save_json(data, file_path):
    with open(file_path, 'w', encoding='utf-8') as file:
        json.dump(data, file, ensure_ascii=False, indent=4)

In [22]:
import re

def get_number_from_string(text):
    # Tìm dấu gạch dưới và các chữ số ở cuối chuỗi
    match = re.search(r'_(\d+)$', text)
    if match:
        return int(match.group(1)) # Trả về số dưới dạng string
    return None

# Test thử
data = ["WIKI_10", "CRAWL_NER_WIKI_25", "VERSION_99"]
results = [get_number_from_string(s) for s in data]
print(results) 

[10, 25, 99]


In [23]:
CACHE_URL_PATH = r"data\cache\mapping\hash2url.json"
CACHE_SEARCH_PATH = r"data\cache\mapping\hash_id_IR_20.json"
METADATA_PATH = r"data\metadata_medhop.json"

BASE_1000 = "data/ALL_ROUND/medhop_results_gg_direct"
DIR_SEARCH = "data/cache/IR_20"
DIR_WIKI = "data/cache/wiki_pages_tool_1"

In [24]:
def flatten_wiki_cache(file_path):
    # 1. Load dữ liệu từ file
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            raw_data = json.load(f)
    except Exception as e:
        print(f"Lỗi khi đọc file: {e}")
        return {}

    final_cache = {}

    # 2. Duyệt qua từng cấp độ của JSON
    # main_id: "157829"
    # tasks: {"crawl_wiki": {...}, "crawl_ner_wiki": {...}}
    for main_id, tasks in raw_data.items():
        for task_name, items in tasks.items():
            # Duyệt qua các cặp hash_id: url trong từng task
            for hash_id, url in items.items():
                # Lưu vào dict chung
                final_cache[hash_id] = url

    return final_cache

FILE_PATH = r'data\cache\all_hash_id_crawl_wiki_embed.json' 
url_mapping = flatten_wiki_cache(FILE_PATH)

print(f"Tổng số HashID thu được: {len(url_mapping)}")

Tổng số HashID thu được: 4074


In [25]:
CACHE_URL_WIKI = load_json(CACHE_URL_PATH)
CACHE_SEARCH = load_json(CACHE_SEARCH_PATH)
metadata = load_json(METADATA_PATH)
    
CACHE_URL_WIKI.update(url_mapping)

In [26]:
len(CACHE_URL_WIKI)

53017

In [27]:
import re

def load_retrieval_map(data_input):
    """
    Map ID (WIKI_10, CRAWL_NER_WIKI_2,...) trực tiếp với hash_id theo thứ tự 1-1.
    """
    result_map = {}
    
    for key, entry in data_input.items():
        crawl_docs = entry.get("crawl_doc", [])
        hash_ids = entry.get("hash_id", [])
        
        for i, doc_text in enumerate(crawl_docs):
            # Regex mới: 
            # \[ : Bắt đầu ngoặc vuông
            # ( : Bắt đầu group muốn lấy
            #  .*? : Mọi ký tự (WIKI, CRAWL_NER_WIKI,...)
            #  _ : Dấu gạch dưới
            #  \d+ : Các chữ số
            # ) : Kết thúc group
            # \] : Đóng ngoặc vuông
            match = re.search(r'\[(.*?_\d+)\]', doc_text)
            
            if match and i < len(hash_ids):
                source_key = match.group(1) # Kết quả: "WIKI_10" hoặc "CRAWL_NER_WIKI_2"
                result_map[source_key] = hash_ids[i]
                
    return result_map

In [28]:
def extract_detail_regex(text):
    pattern = r'(?i)Detailed explanation[:\s]*(.*)'
    match = re.search(pattern, text)
    
    if match:
        return match.group(1).strip()
    return ""

long_answer = "Reference: [GG_5], [GG_6]. Detailed explanation: The oxygen level..."
print(extract_detail_regex(long_answer))

The oxygen level...


```Question: 
===
Subquestion 1: 
Question:
Answer:
Referece:
Detailed explanation:

Subquestion 2:
Question:
Answer:
Referece:
Detailed explanation:
===
Conclusion:
Answer:
Detailed explanation: 
=============
Question:
Answer:
Referece:
Detailed explanation: ```

# DIRECT

In [29]:
BASE_1000 = "data/ALL_ROUND/medhop_results_gg_direct/all_ans"
DIR_SEARCH = "data/cache/IR_20"
DIR_WIKI = "data/cache/wiki_pages_tool_1"



In [30]:
output = {}
for qidx, item in metadata.items():
    if item['type_q'] != 'direct':
        continue
    
    if qidx != "100860":
        continue
    file_path = f"{BASE_1000}/R4_step_3b_all_ans_{qidx}.json"
    tmp_data = load_json(file_path)
    
    final_answer = tmp_data['1']['0']['0']['final_answer']
    if final_answer['selection_status'] != 'selected':
        continue
    
    
    question = tmp_data['1']['0']['0']['question']
    search_kw = tmp_data['1']['0']['0']['search_keyword'].lower().strip()
    
    short_answer = final_answer['short_answer']
    long_answer = final_answer['long_answer']
    detail_explanation = extract_detail_regex(long_answer)
    sources = final_answer['sources']
    
    retrieval_information = tmp_data['1']['0']['0']['retrieval_information']
       
    if search_kw in CACHE_SEARCH:
        hash_id_gg = CACHE_SEARCH[search_kw]
        file_gg = f'{DIR_SEARCH}/{hash_id_gg}'
    else:
        print(f"CACHE 0 co SEARCH GG  {qidx}")
        continue
    
    gg_data = load_json(file_gg)['google']['search_results']
    crawl_wiki_data = load_retrieval_map(retrieval_information)
    
    reference_inf = f"Question: {question}\n Answer: {short_answer}\n Referece:\n "
    
    print(crawl_wiki_data)
    print(sources)
    for src in sources:
        index_link = get_number_from_string(src)
        if index_link:
            index_link = index_link-  1 # start = 0
        else:
            print(qidx)
            print(file_path)
            exit()
            continue
        if 'GG' in src:            
            reference_inf += gg_data[index_link]['link'] + "\n "
        else:
            wiki_hashid = crawl_wiki_data[src]
            file_wiki = f"{DIR_WIKI}/{wiki_hashid}"
            print(file_wiki)
            # if os.path.exists()
            reference_inf += CACHE_URL_WIKI[wiki_hashid] + "\n "   
            
    reference_inf += f"Detailed explanation: {detail_explanation}"
        
    print(reference_inf)
    output[qidx] = reference_inf
    

{'CRAWL_NER_WIKI_1': '094fe6d92e131db04977713946db9618.json', 'CRAWL_NER_WIKI_3': '5437a088679463a9e4f1f63bf5a34779.json', 'CRAWL_NER_WIKI_2': '3f7b93861a7143afa23f29f1bc3a0a8b.json'}
['CRAWL_NER_WIKI_1']
data/cache/wiki_pages_tool_1/094fe6d92e131db04977713946db9618.json
Question: Which non-ionizing imaging technique that can also be used to form images of mummies is used to confirm that the heart and other organs are completely reversed in a patient with situs inversus totalis?
 Answer: mri
 Referece:
 https://en.wikipedia.org/wiki/situs_inversus
 Detailed explanation: The context states that situs inversus is diagnosed using imaging techniques such as x‐ray, ultrasound, CT scan, and magnetic resonance imaging (MRI). Among these, MRI is a non-ionizing imaging method. Furthermore, MRI has been used to form images of mummies, making it the technique that confirms the complete reversal of the heart and other organs in a patient with situs inversus totalis.


In [31]:
save_json(output, 'tmp.json')

# SEQUENCE

In [35]:
output_seq = {}
BASE_SEQ = "SEQ_medhop_results/all_ans"
for qidx, item in metadata.items():
    if item['type_q'] != 'sequence':
        continue
    
    if qidx != "116951":
        continue
    
    file_path = f"{BASE_SEQ}/R4_step_3b_all_ans_{qidx}.json"
    tmp_data = load_json(file_path)
    
    prev_answer = ""
    for step, step_data in tmp_data['1']['0'].items():
        step = str(step)
        print("STEP: ", step)
        final_answer = tmp_data['1']['0'][step]['final_answer']
        question = tmp_data['1']['0'][step]['question']
        search_kw = tmp_data['1']['0'][step]['search_keyword'].lower().strip()
        
        short_answer = final_answer['short_answer']
        long_answer = final_answer['long_answer']
        
        if step != '0':
            search_kw = search_kw + " " + prev_answer

        
        reference_inf = f"Question: {question}\n Answer: {short_answer}\n Referece:\n "
        
        if final_answer['selection_status'] != 'selected':
            reference_inf += "ZEROSHOT - O3"
        else:        
            detail_explanation = extract_detail_regex(long_answer)
            sources = final_answer['sources']
            
            retrieval_information = tmp_data['1']['0'][step]['retrieval_information']
            crawl_wiki_data = load_retrieval_map(retrieval_information)
            
            print(search_kw)
            
            if search_kw in CACHE_SEARCH:
                hash_id_gg = CACHE_SEARCH[search_kw]
                file_gg = f'{DIR_SEARCH}/{hash_id_gg}'
            else:
                print(f"CACHE 0 co SEARCH GG  {qidx}")
                continue
            
            gg_data = load_json(file_gg)['google']['search_results']
            
        
            print(crawl_wiki_data)
            print(sources)
            for src in sources:
                index_link = get_number_from_string(src)
                if index_link:
                    index_link = index_link-  1 # start = 0
                else:
                    # print(qidx)
                    # print(file_path)
                    # exit()
                    continue
                
                if 'GG' in src:            
                    reference_inf += gg_data[index_link]['link'] + "\n "
                else:
                    wiki_hashid = crawl_wiki_data[src]
                    file_wiki = f"{DIR_WIKI}/{wiki_hashid}"
                    print(file_wiki)
                    # if os.path.exists()
                    reference_inf += CACHE_URL_WIKI[wiki_hashid] + "\n "   
                    
            reference_inf += f"Detailed explanation: {detail_explanation}"
                
            print(reference_inf)
            output[qidx] = reference_inf
        
        prev_answer = short_answer.lower()

STEP:  0
condition reactivation virus painful skin rash blisters
{'CRAWL_WIKI_1': 'be939a3de3738b650c5effd551b24949.json', 'CRAWL_WIKI_3': '84d1876f5d17c365821d0837c0dca316.json', 'CRAWL_NER_WIKI_3': '4f994bb5a70695dd1023a38ac4a96bea.json', 'CRAWL_WIKI_2': '99bc1769e9f5cb0afff2e44aac2d363f.json', 'CRAWL_NER_WIKI_1': 'bdef0e86207a9a78bfea391e750b5bc2.json', 'CRAWL_NER_WIKI_2': 'bc57dc7997d985a2c9cc68a8f446c347.json'}
['GG_8', 'GG_3', 'CRAWL_WIKI_2', 'GG_16', 'GG_14', 'CRAWL_WIKI_3', 'GG_11', 'GG_5', 'GG_1', 'GG_4']
data/cache/wiki_pages_tool_1/99bc1769e9f5cb0afff2e44aac2d363f.json
data/cache/wiki_pages_tool_1/84d1876f5d17c365821d0837c0dca316.json
Question: What is the condition caused by reactivation of a virus that usually causes a painful skin rash with blisters?
 Answer: shingles
 Referece:
 https://www.dermatologyaffiliates.com/blog/identifying-early-signs-of-shingles
 https://www.healthdirect.gov.au/shingles
 https://en.wikipedia.org/wiki/varicella_zoster_virus
 https://www.hirslan

# PARALLEL

In [ ]:
output_seq = {}
BASE_PAR = "data/ALL_ROUND/1000/medhop_results_ct6_r36_parallel_seq/all_ans"
for qidx, item in metadata.items():
    if item['type_q'] != 'parallel':
        continue
    
    if qidx != "112756":
        continue
    
    file_path = f"{BASE_PAR}/R4_step_3b_all_ans_{qidx}.json"
    tmp_data = load_json(file_path)
    
    for qi in ["1", "2", "3", "4"]:
        if qi not in tmp_data:
            break
        print('Qi: ', qi)
        prev_answer = ""
        for step, step_data in tmp_data[qi]['0'].items():
            step = str(step)
            print("STEP: ", step)
            final_answer = tmp_data[qi]['0'][step]['final_answer']
            question = tmp_data[qi]['0'][step]['question']
            search_kw = tmp_data[qi]['0'][step]['search_keyword'].lower().strip()
            
            short_answer = final_answer['short_answer']
            long_answer = final_answer['long_answer']
            
            if step != '0':
                search_kw = search_kw + " " + prev_answer

            
            reference_inf = f"Question: {question}\n Answer: {short_answer}\n Referece:\n "
            
            if final_answer['selection_status'] != 'selected':
                reference_inf += "ZEROSHOT - O3"
            else:        
                detail_explanation = extract_detail_regex(long_answer)
                sources = final_answer['sources']
                
                retrieval_information = tmp_data[qi]['0'][step]['retrieval_information']
                crawl_wiki_data = load_retrieval_map(retrieval_information)
                
                print(search_kw)
                
                if search_kw in CACHE_SEARCH:
                    hash_id_gg = CACHE_SEARCH[search_kw]
                    file_gg = f'{DIR_SEARCH}/{hash_id_gg}'
                else:
                    print(f"CACHE 0 co SEARCH GG  {qidx}")
                    continue
                
                gg_data = load_json(file_gg)['google']['search_results']
                
            
                # print(crawl_wiki_data)
                # print(sources)
                for src in sources:
                    index_link = get_number_from_string(src)
                    if index_link:
                        index_link = index_link-  1 # start = 0
                    else:
                        continue
                    
                    if 'GG' in src:            
                        reference_inf += gg_data[index_link]['link'] + "\n "
                    else:
                        wiki_hashid = crawl_wiki_data[src]
                        file_wiki = f"{DIR_WIKI}/{wiki_hashid}"
                        print(file_wiki)
                        # if os.path.exists()
                        reference_inf += CACHE_URL_WIKI[wiki_hashid] + "\n "   
                        
                reference_inf += f"Detailed explanation: {detail_explanation}"
                    
                print(reference_inf)
                output[qidx] = reference_inf
            
            prev_answer = short_answer.lower()

Qi:  1
STEP:  0
cowden syndrome colorectal malignancy risk
{'CRAWL_WIKI_2': '6a38d779dbfde4d5bc9ff63d7f0a609f.json', 'CRAWL_NER_WIKI_1': 'f621b5375572874db8284066140a1bba.json', 'CRAWL_WIKI_1': '3e7af60fddd61b1f19338a1428405a12.json', 'CRAWL_NER_WIKI_4': '3e7af60fddd61b1f19338a1428405a12.json', 'CRAWL_NER_WIKI_3': 'a51d9717aba43ad9e5e50c9ff7947e18.json', 'CRAWL_NER_WIKI_2': 'c6ee3d4a5d429ccca050a46d876297c2.json', 'CRAWL_WIKI_3': '4910905ff53a589356cf38f7bd3a586e.json', 'CRAWL_NER_WIKI_5': '9ff971ed9b390132e83347795cc4abab.json'}
['CRAWL_NER_WIKI_4', 'CRAWL_NER_WIKI_1']
data/cache/wiki_pages_tool_1/3e7af60fddd61b1f19338a1428405a12.json
data/cache/wiki_pages_tool_1/f621b5375572874db8284066140a1bba.json
Question: What is the risk of colorectal malignancy associated with Cowden syndrome?
 Answer: 9%
 Referece:
 https://en.wikipedia.org/wiki/cowden_syndrome
 https://en.wikipedia.org/wiki/colorectal_polyp
 Detailed explanation: The context notes that, in Cowden syndrome, polyps are very com